# LeetCode #1349: Maximum Students Taking Exam

https://leetcode.com/problems/maximum-students-taking-exam/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(2^{m \cdot n})$ | $O(m \cdot n)$ |
| **Optimal: Bitmask DP ★** | $O(m \cdot 4^n)$ | $O(2^n)$ |

---

## Understanding the Methods

### Brute Force
Enumerate every seat assignment for all $m \times n$ seats and check all cheating constraints — exponential.

### Optimal: Bitmask DP ★
Represent each row as a bitmask of seated students. A valid mask must place students only in working seats and have no two adjacent students. Between consecutive rows, students in column $j$ of row $r$ must not sit directly in front of students in columns $j-1$ or $j+1$ of row $r+1$. DP over rows, tracking the maximum student count for each valid mask pair.

**Constraints:**
* `seats` is `m x n` where $1 \leq m \leq 8$, $1 \leq n \leq 8$
* `seats[i][j]` is either `'.'` (good) or `'#'` (broken)

## Solutions

### C#

In [ ]:
public class Solution {
    public int MaxStudents(char[][] seats) {
        int m = seats.Length, n = seats[0].Length;
        // Encode each row's good seats as a bitmask
        int[] goodSeats = new int[m];
        for (int i = 0; i < m; i++)
            for (int j = 0; j < n; j++)
                if (seats[i][j] == '.') goodSeats[i] |= (1 << j);

        // dp[mask] = max students in current row using this column mask
        int[] dp = new int[1 << n];
        int[] prev = new int[1 << n];
        int result = 0;

        for (int i = 0; i < m; i++) {
            int[] curr = new int[1 << n];
            for (int mask = 0; mask < (1 << n); mask++) curr[mask] = -1;

            for (int mask = 0; mask < (1 << n); mask++) {
                // Mask must only seat students in good seats
                if ((mask & goodSeats[i]) != mask) continue;
                // No two students sit adjacent in the same row
                if ((mask & (mask >> 1)) != 0) continue;

                for (int prevMask = 0; prevMask < (1 << n); prevMask++) {
                    if (prev[prevMask] < 0) continue;
                    // No student can see diagonally to the row above
                    if ((mask & (prevMask >> 1)) != 0) continue;
                    if ((mask & (prevMask << 1)) != 0) continue;

                    int students = prev[prevMask] + int.PopCount((uint)mask);
                    if (students > curr[mask]) curr[mask] = students;
                }
                result = Math.Max(result, curr[mask]);
            }
            prev = curr;
        }
        return result;
    }
}

### Python

In [ ]:
class Solution:
    def max_students(self, seats: list[list[str]]) -> int:
        m, n = len(seats), len(seats[0])
        # Encode each row's good seats as a bitmask
        good = [
            sum(1 << j for j in range(n) if seats[i][j] == '.') for i in range(m)
        ]
        # dp[mask] = max students placed when current row uses this mask
        INF = -1
        prev = {0: 0}  # previous row: mask -> student count

        result = 0
        for i in range(m):
            curr = {}
            for mask in range(1 << n):
                # Must place only in good seats and no adjacent pair
                if (mask & good[i]) != mask: continue
                if mask & (mask >> 1): continue

                for prev_mask, prev_count in prev.items():
                    # No diagonal cheating from the row above
                    if mask & (prev_mask >> 1): continue
                    if mask & (prev_mask << 1): continue
                    count = prev_count + bin(mask).count('1')
                    if mask not in curr or curr[mask] < count:
                        curr[mask] = count
                    result = max(result, curr[mask])

            prev = curr if curr else {0: 0}

        return result

### Go

In [ ]:
import "math/bits"

func maxStudents(seats [][]byte) int {
    m, n := len(seats), len(seats[0])
    // Encode each row's good seats as a bitmask
    goodSeats := make([]int, m)
    for i := 0; i < m; i++ {
        for j := 0; j < n; j++ {
            if seats[i][j] == '.' { goodSeats[i] |= 1 << j }
        }
    }

    size := 1 << n
    prev := make([]int, size)
    for i := range prev { prev[i] = -1 }
    prev[0] = 0
    result := 0

    for i := 0; i < m; i++ {
        curr := make([]int, size)
        for j := range curr { curr[j] = -1 }

        for mask := 0; mask < size; mask++ {
            if mask & goodSeats[i] != mask { continue }
            if mask & (mask >> 1) != 0 { continue }

            for prevMask := 0; prevMask < size; prevMask++ {
                if prev[prevMask] < 0 { continue }
                if mask & (prevMask >> 1) != 0 { continue }
                if mask & (prevMask << 1) != 0 { continue }

                students := prev[prevMask] + bits.OnesCount(uint(mask))
                if students > curr[mask] { curr[mask] = students }
            }
            if curr[mask] > result { result = curr[mask] }
        }
        prev = curr
    }
    return result
}

### Rust

In [ ]:
impl Solution {
    pub fn max_students(seats: Vec<Vec<char>>) -> i32 {
        let (m, n) = (seats.len(), seats[0].len());
        // Encode each row's good seats as a bitmask
        let good: Vec<usize> = (0..m).map(|i| {
            (0..n).filter(|&j| seats[i][j] == '.').fold(0, |acc, j| acc | (1 << j))
        }).collect();

        let size = 1 << n;
        let mut prev = vec![-1i32; size];
        prev[0] = 0;
        let mut result = 0;

        for i in 0..m {
            let mut curr = vec![-1i32; size];
            for mask in 0..size {
                if (mask & good[i]) != mask { continue; }
                if (mask & (mask >> 1)) != 0 { continue; }
                for prev_mask in 0..size {
                    if prev[prev_mask] < 0 { continue; }
                    if (mask & (prev_mask >> 1)) != 0 { continue; }
                    if (mask & (prev_mask << 1)) != 0 { continue; }
                    let students = prev[prev_mask] + mask.count_ones() as i32;
                    if students > curr[mask] { curr[mask] = students; }
                }
                if curr[mask] > result { result = curr[mask]; }
            }
            prev = curr;
        }
        result
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `seats = [["#",".","#","#",".","#"],[".","#","#","#","#","."],["#",".","#","#",".","#"]]`
With broken seats blocking most positions, the maximum non-cheating arrangement yields $4$ students.

### 2. Slightly Complex
**Input:** `seats = [[".","#"],["#","#"],["#","."],["#","#"],[".","#"]]`
Only 3 seats are available across the grid. No pair of good seats triggers a cheating constraint, so all 3 can be used.

### 3. Edge Case: Time Factor
**Input:** Full $8 \times 8$ grid of good seats
$2^8 = 256$ masks per row; the DP evaluates $256 \times 256 = 65{,}536$ mask pairs per row and $8$ rows — about $500{,}000$ operations total.

### 4. Edge Case: Space Factor
**Input:** $8 \times 8$ grid
The previous-row dp array has $2^8 = 256$ entries. Rolling the DP means only two arrays of size 256 are live at once — $O(2^n)$ space.

### 5. Almost-Impossible but Plausible
**Input:** $8 \times 8$ all-good seats in a checkerboard pattern
Students can only sit in alternating seats. The bitmask DP evaluates every valid mask, correctly rejecting adjacent-in-row and diagonal-across-rows placements, returning the true maximum.